In [3]:
"""
Dataset Generator for Campus Coffee Shop Assignment.
Creates nested folders, mixed encoding logs, and nested config JSON.
"""

import json
import random
from datetime import datetime, timedelta
from pathlib import Path

BASE_DIR = Path("C:\\", "Users", "tutor5923", "Documents", "folder_for_saving", "d2_data")  # Path("data")
STORES = ["library_cafe", "science_building", "student_union"]
REGISTERS = ["POS_01", "POS_02", "POS_03"]
CURRENCIES = ["EUR", "NOK", "USD"]
ITEMS = [
    ("Espresso", 3.00),
    ("Latte", 4.50),
    ("Cappuccino", 4.20),
    ("Muffin", 3.50),
    ("Croissant", 3.80),
    ("Sandwich", 7.50),
]
STATUSES = ["SUCCESS", "SUCCESS", "SUCCESS", "SUCCESS", "REFUND", "CANCELLED"]

NOISE_LINES = [
    "--- PRINTER PAPER LOW ---",
    "[DEBUG] BARCODE SCANNER REBOOTED",
    "=== NETWORK SYNC OK ===",
]


def generate_log_line(ts: datetime, reg_id: str) -> str:
    item_name, base_price = random.choice(ITEMS)
    currency = random.choice(CURRENCIES)

    price = 999.00 if random.random() < 0.03 else round(base_price + random.uniform(-0.5, 0.5), 2)
    status = random.choice(STATUSES)
    ts_str = ts.strftime("%Y-%m-%d %H:%M:%S")

    return f"[{ts_str}] [{reg_id}] item={item_name} price={price:.2f} {currency} ({status})\n"


def create_dataset():
    start_time = datetime(2026, 10, 15, 7, 0, 0)
    file_idx = 0

    for store in STORES:
        for reg in REGISTERS:
            folder = BASE_DIR / "2026" / store / reg
            folder.mkdir(parents=True, exist_ok=True)

            log_file = folder / "register_sales.log"
            encoding = "utf-8" if file_idx % 2 == 0 else "iso-8859-1"
            
            lines = []
            current_time = start_time

            for _ in range(80):
                current_time += timedelta(seconds=random.randint(10, 180))
                if random.random() < 0.15:
                    lines.append(random.choice(NOISE_LINES) + "\n")
                lines.append(generate_log_line(current_time, reg))

            with open(log_file, "w", encoding=encoding) as f:
                f.writelines(lines)

            file_idx += 1
            print(f"Generated: {log_file} [{encoding.upper()}]")

    # Generate nested JSON config
    config = {
        "company": "Campus Coffee",
        "regions": [
            {
                "region_name": "Main Campus",
                "stores": [
                    {
                        "name": "Library Cafe",
                        "registers": [
                            {"id": "POS_01", "active": True},
                            {"id": "POS_02", "active": True},
                        ],
                    },
                    {
                        "name": "Science Building",
                        "registers": [
                            {"id": "POS_03", "active": True},
                            {"id": "POS_04", "active": False},
                        ],
                    },
                ],
            }
        ],
    }

    with open(BASE_DIR / "store_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    print("\nDataset generation complete!")


if __name__ == "__main__":
    create_dataset()


PermissionError: [WinError 5] Ingen tilgang: 'C:\\Users\\tutor5923'

task 3

In [ ]:
import numpy as np
import pandas as pd


# Task 3.1: Fjerning av pris-outliers

def remove_price_outliers(prices):

#Konverterer prisdata til et 1D NumPy-array og erstatter feilregistreringer

    # Konverterer til 1D NumPy float-array
    prices_array = np.array(prices, dtype=float)
    
    # Beregner gjennomsnitt (mu) og standardavvik (sigma)
    mean = np.mean(prices_array)
    std = np.std(prices_array)
    
    # Betingelse: |x - mu| > 3 * sigma
    outliers_mask = np.abs(prices_array - mean) > 3 * std
    
    # Beregner medianen til de gyldige prisene (der det IKKE er outlier)
    valid_prices = prices_array[~outliers_mask]
    valid_median = np.median(valid_prices)
    
    # Lager en kopi og erstatter outliers med medianen
    cleaned_prices = prices_array.copy()
    cleaned_prices[outliers_mask] = valid_median
    
    return cleaned_prices



# Task 3.2: Beregning av refusjonsrate

def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """
    Beregner elementvis refund ratio ved hjelp av NumPy array-operasjoner:
    Refund Ratio = Refund Count / (Sales Count + 1e-6)
    Legger til epsilon (1e-6) for å unngå 'division by zero'-feil.
    """
    epsilon = 1e-6
    refund_ratio = refund_count / (sales_count + epsilon)
    return refund_ratio

task 4

In [ ]:
# Task 4.1: Opprette DataFrame fra transaksjonsliste

def create_transaction_dataframe(transactions):
    """
    Konverterer en liste med transaksjons-dictionaries til en Pandas DataFrame,
    setter timestamp-kolonnen til datetime-format, setter den som indeks og
    sorterer DataFrame-en kronologisk.
    """
    df = pd.DataFrame(transactions)
    
    # Konverterer timestamp til datetime og setter som indeks
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp')
    
    # Sorterer kronologisk
    df = df.sort_index()
    return df



# Task 4.2: Resampling av totalinntekt per time per vare

def calculate_hourly_revenue(df):
    """
    Grupperer data etter vare ('item') og resampler transaksjonene i 
    1-times tidsvinduer ('1h') for å beregne samlet inntekt (sum of price).
    """
    # Grupperer på 'item', resampler indeksen (timestamp) per time, og summerer prisen
    hourly_rev = df.groupby('item')['price'].resample('1h').sum().reset_index()
    return hourly_rev



# Task 4.3: Pivot-tabell over omsetning fordelt på status

def create_status_pivot_table(df):
    """
    Lager en pivot-tabell som viser samlet transaksjonsomsetning fordelt på
    vare ('item') langs radene og status (SUCCESS, REFUND, CANCELLED) langs kolonnene.
    """
    pivot = pd.pivot_table(
        df,
        values='price',
        index='item',
        columns='status',
        aggfunc='sum',
        fill_value=0.0
    )
    return pivot



# Task 4.4: Beregne refund ratio per vare med NumPy-funksjonen

def calculate_item_refund_ratios(df):
    """
    Beregner refusjonsrate per vare ved å telle antall vellykkede salg ('SUCCESS')
    og antall refusjoner ('REFUND'), og benytter compute_refund_ratio fra Task 3.2.
    """
    # Teller antall forekomster per item og status
    status_counts = df.groupby(['item', 'status']).size().unstack(fill_value=0)
    
    # Henter ut antall salg og refusjoner som NumPy-arrays
    sales_count = status_counts['SUCCESS'].to_numpy() if 'SUCCESS' in status_counts.columns else np.zeros(len(status_counts))
    refund_count = status_counts['REFUND'].to_numpy() if 'REFUND' in status_counts.columns else np.zeros(len(status_counts))
    
    # Kaller NumPy-funksjonen fra Task 3.2
    ratios = compute_refund_ratio(sales_count, refund_count)
    
    # Samler resultatene i en oversiktlig DataFrame
    refund_ratio_df = pd.DataFrame({
        'item': status_counts.index,
        'sales_count': sales_count,
        'refund_count': refund_count,
        'refund_ratio': ratios
    }).set_index('item')
    
    return refund_ratio_df